In [1]:
from pathlib import Path
from datetime import timedelta
import sys
import time
import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb
import plotly.express as px

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
PROC = ROOT / "data" / "processed"
FIG_DIR = ROOT / "reports" / "figures"
REP = ROOT / "reports"
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))
from features import construir_features, FEATS, CAT_FEATS
from simulate import simular_rs

# --- Supuestos de la política (R, S) ---
R, L = 7, 7               # revisión semanal, lead time de 7 días
PROT = R + L - 1          # días que cubre cada pedido: del día siguiente hasta antes del próximo arribo
N_CALIB, N_SIM = 13, 26   # semanas de calibración y de simulación
H_ANUAL = 0.25            # costo de mantener: 25% anual del precio
MARGEN = 0.30             # costo de faltante: margen perdido por unidad no vendida
WARMUP = 14               # días iniciales que no se miden
ALPHAS = [0.50, 0.70, 0.80, 0.90, 0.95, 0.98, 0.99]

df = pl.read_parquet(PROC / "m5_CA_1.parquet")
seg = pl.read_parquet(PROC / "segmentacion_CA_1.parquet")
mascara = pl.read_parquet(PROC / "mascara_CA_1.parquet")
ids_validos = (pd.read_parquet(PROC / "cv_baselines_CA_1.parquet", columns=["unique_id"])
                 ["unique_id"].unique().tolist())

# --- Calendario (todo antes de la prueba final reservada) ---
dev_end = pd.Timestamp(df["date"].max()) - timedelta(days=28)
last_review = dev_end - timedelta(days=PROT)
reviews_sim = [last_review - timedelta(days=7 * k) for k in range(N_SIM)][::-1]
sim_start = reviews_sim[0]
last_calib = sim_start - timedelta(days=PROT + 1)
origins_calib = [last_calib - timedelta(days=7 * k) for k in range(N_CALIB)][::-1]
train_end = origins_calib[0]
origenes = pd.date_range(origins_calib[0], last_review, freq="7D")
N_WINDOWS = len(origenes)

print("Entrenamiento LightGBM hasta:", train_end.date())
print("Calibración (orígenes):      ", origins_calib[0].date(), "→", last_calib.date(), f"({N_CALIB})")
print("Simulación (revisiones):     ", sim_start.date(), "→", last_review.date(), f"({N_SIM})")
print("Último día simulado:         ", dev_end.date(), f"| orígenes totales: {N_WINDOWS}")

Entrenamiento LightGBM hasta: 2015-07-13
Calibración (orígenes):       2015-07-13 → 2015-10-05 (13)
Simulación (revisiones):      2015-10-19 → 2016-04-11 (26)
Último día simulado:          2016-04-24 | orígenes totales: 40


In [2]:
dev = (df.filter(pl.col("date") <= dev_end.date())
         .with_columns(pl.col("id").cast(pl.Utf8))
         .filter(pl.col("id").is_in(ids_validos)))

inicio = dev.group_by("id").agg(inicio=pl.col("date").min())
fuera = (mascara.filter((pl.col("date") > train_end.date())
                        & pl.col("tipo").is_in(["inactivo", "pre_lanzamiento"]))
                ["id"].unique().to_list())
universo = sorted(
    inicio.filter(pl.col("inicio") <= (train_end - timedelta(days=112)).date())
          .filter(~pl.col("id").is_in(fuera))["id"].to_list()
)
print(f"Productos simulados: {len(universo):,} de {len(ids_validos):,} "
      f"(excluidos por inactividad en el periodo: {len(fuera):,})")

d = (
    dev.filter(pl.col("id").is_in(universo))
    .join(mascara.select(["id", "date", "tipo", "excluir"]), on=["id", "date"], how="left")
    .with_columns(pl.col("excluir").fill_null(False), pl.col("tipo").fill_null("normal"),
                  ventas=pl.col("sales").cast(pl.Float64))
    .sort(["id", "date"])
    .with_columns(limpia=pl.when(pl.col("excluir")).then(pl.lit(None)).otherwise(pl.col("ventas")))
    .with_columns(
        num=pl.col("limpia").fill_null(0).rolling_sum(28).shift(1).over("id"),
        den=(~pl.col("excluir")).cast(pl.Float64).rolling_sum(28).shift(1).over("id"),
        media_prod=pl.col("limpia").mean().over("id"),
    )
    .with_columns(imp=pl.when(pl.col("den") > 0).then(pl.col("num") / pl.col("den"))
                        .otherwise(pl.col("media_prod")))
    .with_columns(demanda=pl.when(pl.col("tipo") == "agotado").then(pl.col("imp"))
                            .otherwise(pl.col("ventas")))
    # Navidad: la tienda cierra, la demanda real es 0
    .with_columns(demanda=pl.when((pl.col("date").dt.month() == 12) & (pl.col("date").dt.day() == 25))
                            .then(0.0).otherwise(pl.col("demanda")))
)

ventana = d.filter(pl.col("date") > train_end.date())
print(f"Días-producto con demanda imputada (agotado) en calibración + simulación: "
      f"{(ventana['tipo'] == 'agotado').mean():.1%}")

Productos simulados: 2,467 de 3,041 (excluidos por inactividad en el periodo: 569)


Días-producto con demanda imputada (agotado) en calibración + simulación: 2.6%


In [3]:
from statsforecast import StatsForecast
from statsforecast.models import SeasonalNaive, TSB

Y = d.select(unique_id="id", ds="date", y=pl.col("ventas").cast(pl.Float32)).to_pandas()
Y["ds"] = pd.to_datetime(Y["ds"]).astype("datetime64[ns]")

t0 = time.time()
sf = StatsForecast(models=[SeasonalNaive(season_length=7), TSB(alpha_d=0.1, alpha_p=0.1)],
                   freq="D", n_jobs=-1)
cv_sf = sf.cross_validation(df=Y, h=PROT, step_size=7, n_windows=N_WINDOWS)
if "unique_id" not in cv_sf.columns:
    cv_sf = cv_sf.reset_index()
cv_sf["cutoff"] = pd.to_datetime(cv_sf["cutoff"]).astype("datetime64[ns]")
cv_sf[["SeasonalNaive", "TSB"]] = cv_sf[["SeasonalNaive", "TSB"]].clip(lower=0)
print(f"Tiempo: {(time.time() - t0) / 60:.1f} min")

assert cv_sf["cutoff"].min() == origenes[0] and cv_sf["cutoff"].max() == origenes[-1], "Orígenes no coinciden"
print("Orígenes:", cv_sf["cutoff"].nunique(), "| primero:", cv_sf["cutoff"].min().date(),
      "| último:", cv_sf["cutoff"].max().date())

Tiempo: 0.1 min
Orígenes: 40 | primero: 2015-07-13 | último: 2016-04-11


In [4]:
PARAMS = dict(objective="tweedie", tweedie_variance_power=1.1,
              n_estimators=500, learning_rate=0.05, num_leaves=127, min_child_samples=100,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
              n_jobs=-1, verbose=-1)

pdf = construir_features(dev, mascara)
tr = pdf[(pdf["date"] <= train_end) & (~pdf["excluir"]) & (pdf["navidad"] == 0)]

t0 = time.time()
modelo = lgb.LGBMRegressor(**PARAMS)
modelo.fit(tr[FEATS], tr["sales"], categorical_feature=CAT_FEATS)

te = pdf[(pdf["date"] > train_end) & (pdf["id"].isin(universo))].copy()
te["LGB"] = np.clip(modelo.predict(te[FEATS]), 0, None)
P_lgb = te.pivot(index="date", columns="id", values="LGB").reindex(columns=universo)
print(f"Entrenado con {len(tr):,} filas en {time.time() - t0:.0f} s | pronósticos: {P_lgb.shape}")

Entrenado con 3,317,042 filas en 71 s | pronósticos: (286, 2467)


In [5]:
fechas = pd.date_range(origenes[0] + timedelta(days=1), dev_end)
pdd = d.select(["id", "date", "demanda", "sell_price"]).to_pandas()
pdd["date"] = pd.to_datetime(pdd["date"]).astype("datetime64[ns]")
D_full = pdd.pivot(index="date", columns="id", values="demanda").reindex(index=fechas, columns=universo)
precio_full = pdd.pivot(index="date", columns="id", values="sell_price").reindex(index=fechas, columns=universo)
P_lgb = P_lgb.reindex(index=fechas)
assert not D_full.isna().any().any(), "Huecos en la matriz de demanda"
assert not P_lgb.isna().any().any(), "Huecos en los pronósticos de LightGBM"

def suma_por_origen(M):
    """Suma de los días o+1 … o+PROT para cada origen o."""
    return pd.DataFrame({o: M.loc[o + timedelta(days=1): o + timedelta(days=PROT)].sum()
                         for o in origenes}).T

A = suma_por_origen(D_full)
F = {"LGB": suma_por_origen(P_lgb)}
for m in ["SeasonalNaive", "TSB"]:
    F[m] = (cv_sf.groupby(["cutoff", "unique_id"])[m].sum().unstack()
                 .reindex(index=origenes, columns=universo))
    assert not F[m].isna().any().any(), f"Huecos en {m}"

calib = [o for o in origenes if o <= last_calib]
simo = [o for o in origenes if o >= sim_start]

sesgo = pd.DataFrame({
    m: {"Sesgo_calibración_%": (Fm.loc[calib] - A.loc[calib]).sum().sum() / A.loc[calib].sum().sum() * 100,
        "Sesgo_simulación_%": (Fm.loc[simo] - A.loc[simo]).sum().sum() / A.loc[simo].sum().sum() * 100}
    for m, Fm in F.items()
}).T
sesgo.round(2)   # + = sobrepronóstico de la demanda acumulada a 13 días

,Sesgo_calibración_%,Sesgo_simulación_%
LGB,-2.18,0.39
SeasonalNaive,-5.81,-4.64
TSB,-4.03,-2.49


In [6]:
clase = (seg.select(id=pl.col("id").cast(pl.Utf8), sb_class="sb_class").to_pandas()
            .set_index("id")["sb_class"].reindex(universo))

q = {}
for m, Fm in F.items():
    z = (A.loc[calib] - Fm.loc[calib]) / np.sqrt(Fm.loc[calib] + 1)
    zl = z.T.rename_axis(index="id", columns="origen").stack().rename("z").reset_index()
    zl["clase"] = zl["id"].map(clase)
    q[m] = zl.groupby("clase")["z"].quantile(ALPHAS).unstack()

# Cuantil al 95% por modelo y clase: un valor mayor = más stock de seguridad necesario
pd.DataFrame({m: q[m][0.95] for m in q}).round(2)

,LGB,SeasonalNaive,TSB
clase,,,
erratic,7.31,22.00,10.98
intermittent,2.57,6.00,3.33
lumpy,4.38,12.00,5.82
smooth,3.74,7.58,5.33


In [7]:
sim_fechas = pd.date_range(sim_start, dev_end)
D_sim = D_full.loc[sim_fechas].to_numpy()
precio = precio_full.loc[sim_fechas].to_numpy()
revision = sim_fechas.isin(reviews_sim)
idx_rev = np.where(revision)[0]
assert idx_rev[0] == 0 and len(idx_rev) == N_SIM

def metricas(res, cols=slice(None)):
    s = slice(WARMUP, None)
    dem, vend, perd, inv, p = (x[s][:, cols] for x in
                               (D_sim, res["vendido"], res["perdido"], res["inventario"], precio))
    c_mant = (inv * p * H_ANUAL / 365).sum()
    c_falt = (perd * p * MARGEN).sum()
    return {"fill_rate": vend.sum() / dem.sum(),
            "inv_valor_prom": (inv * p).sum(axis=1).mean(),
            "dias_cobertura": inv.sum() / dem.sum(),
            "costo_mantener": c_mant,
            "costo_faltante": c_falt,
            "costo_total": c_mant + c_falt}

filas, guardadas = [], {}
for m, Fm_all in F.items():
    Fm = Fm_all.loc[reviews_sim].to_numpy()
    for a in ALPHAS:
        qa = clase.map(q[m][a]).to_numpy()
        S = np.zeros_like(D_sim)
        S[idx_rev] = np.maximum(0.0, Fm + qa * np.sqrt(Fm + 1))
        res = simular_rs(D_sim, S, revision, L, inv_inicial=np.ceil(S[0]))
        filas.append({"modelo": m, "alpha": a, **metricas(res)})
        if a == 0.95:
            guardadas[m] = res

tabla = pd.DataFrame(filas)
tabla.to_csv(REP / "simulacion_inventario_CA_1.csv", index=False)
tabla.round({"fill_rate": 4, "inv_valor_prom": 0, "dias_cobertura": 2,
             "costo_mantener": 0, "costo_faltante": 0, "costo_total": 0})

,modelo,alpha,fill_rate,inv_valor_prom,dias_cobertura,costo_mantener,costo_faltante,costo_total
0,LGB,0.50,0.8481,62751.0,4.90,7522.0,103056.0,110577.0
1,LGB,0.70,0.9039,85048.0,6.46,10194.0,61801.0,71995.0
2,LGB,0.80,0.9306,102369.0,7.68,12270.0,42888.0,55159.0
3,LGB,0.90,0.9589,132789.0,9.90,15917.0,24257.0,40174.0
4,LGB,0.95,0.9755,166777.0,12.46,19990.0,13968.0,33958.0
5,LGB,0.98,0.9882,219722.0,16.54,26337.0,6339.0,32676.0
6,LGB,0.99,0.9935,264165.0,20.05,31664.0,3382.0,35045.0
7,SeasonalNaive,0.50,0.8233,87051.0,6.57,10434.0,112240.0,122674.0
8,SeasonalNaive,0.70,0.9019,132851.0,9.55,15924.0,53036.0,68960.0
9,SeasonalNaive,0.80,0.9320,168216.0,11.94,20163.0,33481.0,53644.0


In [8]:
fig = px.line(tabla, x="inv_valor_prom", y="fill_rate", color="modelo", markers=True,
              hover_data=["alpha", "costo_total"],
              labels={"inv_valor_prom": "Inventario promedio (USD)", "fill_rate": "Fill rate",
                      "modelo": "Pronóstico"},
              title="Inventario vs. nivel de servicio — CA_1, 26 revisiones semanales (calibración por clase)")
fig.update_layout(width=1000, height=600, yaxis_tickformat=".0%")
fig.show()

# ¿Cuánto inventario necesita LightGBM para el mismo fill rate que TSB con α = 0.95?
ref = tabla.query("modelo == 'TSB' and alpha == 0.95").iloc[0]
curva = tabla.query("modelo == 'LGB'").sort_values("fill_rate")
if not (curva["fill_rate"].min() <= ref["fill_rate"] <= curva["fill_rate"].max()):
    print("⚠ El fill rate de referencia está fuera del rango de LightGBM; la interpolación se recorta.")
inv_lgb = np.interp(ref["fill_rate"], curva["fill_rate"], curva["inv_valor_prom"])
print(f"Fill rate de referencia (TSB, α=0.95): {ref['fill_rate']:.2%}")
print(f"Inventario promedio — TSB: ${ref['inv_valor_prom']:,.0f} | LightGBM al mismo fill rate: "
      f"${inv_lgb:,.0f} ({inv_lgb / ref['inv_valor_prom'] - 1:+.1%})")

# Nivel de servicio que minimiza el costo total para cada modelo
display(tabla.loc[tabla.groupby("modelo")["costo_total"].idxmin()].round(3))

Fill rate de referencia (TSB, α=0.95): 96.51%
Inventario promedio — TSB: $215,102 | LightGBM al mismo fill rate: $145,615 (-32.3%)


,modelo,alpha,fill_rate,inv_valor_prom,dias_cobertura,costo_mantener,costo_faltante,costo_total
5,LGB,0.98,0.988,219722.093,16.541,26336.552,6339.182,32675.735
10,SeasonalNaive,0.90,0.960,247853.192,17.585,29708.431,18083.961,47792.392
18,TSB,0.95,0.965,215102.476,15.694,25782.831,15538.059,41320.890


In [9]:
abc = (seg.select(id=pl.col("id").cast(pl.Utf8), abc="abc").to_pandas()
          .set_index("id")["abc"].reindex(universo).to_numpy())

por_abc = pd.DataFrame([{"modelo": m, "abc": g, **metricas(res, cols=(abc == g))}
                        for m, res in guardadas.items() for g in ["A", "B", "C"]])
por_abc.to_csv(REP / "simulacion_por_abc_CA_1.csv", index=False)
por_abc.pivot(index="abc", columns="modelo", values=["fill_rate", "inv_valor_prom", "costo_total"]).round(3)

fill_rate                      inv_valor_prom                \
modelo       LGB SeasonalNaive    TSB            LGB SeasonalNaive   
abc                                                                  
A          0.975         0.969  0.961     111730.108    235375.873   
B          0.978         0.989  0.977      35180.092     92367.501   
C          0.978         0.990  0.974      19866.437     56739.290   

                   costo_total                           
modelo         TSB         LGB SeasonalNaive        TSB  
abc                                                      
A       139600.755   24429.833     37564.092  29806.683  
B        47553.233    6210.040     11936.760   7475.385  
C        27948.488    3318.614      7064.352   4038.821

In [10]:
# --- Calibración final: clase de demanda × quintil de volumen pronosticado ---
# Con la calibración solo por clase, un mismo cuantil z se aplica a productos de 0.5 y de 50 u./día.
# Separar por volumen ajusta el stock de seguridad al error real de cada nivel de venta.
N_BINS = 5

def calibrar_por_volumen(Fm_all):
    """Cuantiles de z por (clase, quintil de volumen) con los errores fuera de muestra de la calibración."""
    Fc = Fm_all.loc[calib]
    z = (A.loc[calib] - Fc) / np.sqrt(Fc + 1)
    zl = pd.DataFrame({"id": np.tile(universo, len(calib)),
                       "f": Fc.to_numpy().ravel(), "z": z.to_numpy().ravel()})
    zl["clase"] = zl["id"].map(clase)
    edges = np.unique(np.quantile(zl["f"], np.linspace(0, 1, N_BINS + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    zl["bin"] = pd.cut(zl["f"], edges, labels=False).astype(int)
    return zl.groupby(["clase", "bin"])["z"].quantile(ALPHAS).unstack(), edges

filas_v = []
for m, Fm_all in F.items():
    tabla_q, edges = calibrar_por_volumen(Fm_all)
    Fm = Fm_all.loc[reviews_sim].to_numpy()
    bins = pd.cut(Fm.ravel(), edges, labels=False).astype(int)
    claves = pd.MultiIndex.from_arrays([np.tile(clase.to_numpy(), len(reviews_sim)), bins])
    for a in ALPHAS:
        qa = tabla_q[a].reindex(claves).to_numpy().reshape(Fm.shape)
        # Combinación clase × quintil sin datos de calibración → cuantil de la clase
        qa = np.where(np.isnan(qa), np.tile(clase.map(q[m][a]).to_numpy(), (len(reviews_sim), 1)), qa)
        S = np.zeros_like(D_sim)
        S[idx_rev] = np.maximum(0.0, Fm + qa * np.sqrt(Fm + 1))
        res = simular_rs(D_sim, S, revision, L, inv_inicial=np.ceil(S[0]))
        filas_v.append({"modelo": m, "alpha": a, **metricas(res)})

tabla_v = pd.DataFrame(filas_v)
tabla_v.to_csv(REP / "simulacion_inventario_volumen_CA_1.csv", index=False)

fig = px.line(tabla_v, x="inv_valor_prom", y="fill_rate", color="modelo", markers=True,
              hover_data=["alpha", "costo_total"],
              labels={"inv_valor_prom": "Inventario promedio (USD)", "fill_rate": "Fill rate",
                      "modelo": "Pronóstico"},
              title="Inventario vs. nivel de servicio — CA_1, 26 revisiones semanales (calibración clase × volumen)")
fig.update_layout(width=1000, height=600, yaxis_tickformat=".0%")
fig.write_html(FIG_DIR / "08_tradeoff_inventario.html", include_plotlyjs="cdn")
try:
    fig.write_image(FIG_DIR / "08_tradeoff_inventario.png", scale=2)
except Exception as e:
    print(f"PNG no generado ({e})")
fig.show()

# Inventario al mismo fill rate (95%) y costo en el nivel de servicio óptimo de cada modelo
def inv_en(modelo, fr):
    c = tabla_v.query("modelo == @modelo").sort_values("fill_rate")
    return np.interp(fr, c["fill_rate"], c["inv_valor_prom"])

i_lgb, i_tsb = inv_en("LGB", 0.95), inv_en("TSB", 0.95)
print(f"Fill rate 95% — TSB: ${i_tsb:,.0f} | LightGBM: ${i_lgb:,.0f} ({i_lgb / i_tsb - 1:+.1%})")
opt_v = tabla_v.loc[tabla_v.groupby("modelo")["costo_total"].idxmin()].set_index("modelo")
print(f"Costo óptimo — LightGBM vs. TSB: {opt_v.loc['LGB', 'costo_total'] / opt_v.loc['TSB', 'costo_total'] - 1:+.1%} | "
      f"vs. SeasonalNaive: {opt_v.loc['LGB', 'costo_total'] / opt_v.loc['SeasonalNaive', 'costo_total'] - 1:+.1%}")
display(opt_v.round(3))

Fill rate 95% — TSB: $161,137 | LightGBM: $120,675 (-25.1%)
Costo óptimo — LightGBM vs. TSB: -26.1% | vs. SeasonalNaive: -36.0%


,alpha,fill_rate,inv_valor_prom,dias_cobertura,costo_mantener,costo_faltante,costo_total
modelo,,,,,,,
LGB,0.98,0.990,222312.290,16.913,26647.021,5329.780,31976.801
SeasonalNaive,0.90,0.955,210966.781,15.210,25287.114,24708.305,49995.419
TSB,0.95,0.969,230686.802,16.380,27650.815,15621.998,43272.814


In [11]:
import json

OUT = Path.home() / "Documents" / "portfolio" / "public" / "data" / "m5"
(OUT / "productos").mkdir(parents=True, exist_ok=True)
ALPHAS_WEB = [0.80, 0.90, 0.95, 0.98]
MODELOS_WEB = ["LGB", "TSB"]

# --- Muestra: los 3 productos de mayor ingreso por segmento ABC × clase de demanda ---
segp = (seg.with_columns(pl.col("id").cast(pl.Utf8))
           .filter(pl.col("id").is_in(universo)).to_pandas())
muestra = (segp.sort_values("revenue", ascending=False)
               .groupby(["abc", "sb_class"]).head(3)
               .sort_values(["abc", "sb_class", "revenue"], ascending=[True, True, False])
               .reset_index(drop=True))
pos = {p: i for i, p in enumerate(universo)}
cols = np.array([pos[i] for i in muestra["id"]])
D_m, P_m = D_sim[:, cols], precio[:, cols]

# --- Simulación de la muestra con la calibración por clase × volumen (idéntica a la Celda 10) ---
sim_res = {}
for m in MODELOS_WEB:
    tabla_q, edges = calibrar_por_volumen(F[m])
    Fm = F[m].loc[reviews_sim].to_numpy()[:, cols]
    bins = pd.cut(Fm.ravel(), edges, labels=False).astype(int)
    claves = pd.MultiIndex.from_arrays([np.tile(clase.to_numpy()[cols], len(reviews_sim)), bins])
    for a in ALPHAS_WEB:
        qa = tabla_q[a].reindex(claves).to_numpy().reshape(Fm.shape)
        qa = np.where(np.isnan(qa), clase.iloc[cols].map(q[m][a]).to_numpy(), qa)
        S = np.zeros_like(D_m)
        S[idx_rev] = np.maximum(0.0, Fm + qa * np.sqrt(Fm + 1))
        sim_res[(m, a)] = simular_rs(D_m, S, revision, L, inv_inicial=np.ceil(S[0]))

# --- Pronósticos diarios: LightGBM directo; TSB desde la revisión más reciente que cubre cada día ---
lgb_m = P_lgb.loc[sim_fechas].to_numpy()[:, cols]
tsb_m = (cv_sf[cv_sf["unique_id"].isin(muestra["id"])]
         .sort_values("cutoff").groupby(["unique_id", "ds"])["TSB"].last().unstack(0)
         .reindex(index=sim_fechas, columns=muestra["id"]).to_numpy())
assert not np.isnan(tsb_m).any(), "Faltan pronósticos de TSB en algún día"

def r(x, n=2):
    return [round(float(v), n) for v in x]

s = slice(WARMUP, None)
indice = []
for j, row in muestra.iterrows():
    item = row["id"].replace("_CA_1_evaluation", "")
    simulacion = {}
    for m in MODELOS_WEB:
        simulacion[m] = {}
        for a in ALPHAS_WEB:
            res = sim_res[(m, a)]
            dem = D_m[s, j].sum()
            simulacion[m][f"{a:.2f}"] = {
                "inventario": r(res["inventario"][:, j], 0),
                "perdido": r(res["perdido"][:, j]),
                "pedido": r(res["pedidos"][:, j], 0),
                "fill_rate": round(float(res["vendido"][s, j].sum() / dem), 4) if dem > 0 else None,
                "inv_prom_unidades": round(float(res["inventario"][s, j].mean()), 1),
            }
    datos = {
        "id": item,
        "departamento": item.rsplit("_", 1)[0],
        "abc": row["abc"],
        "clase": row["sb_class"],
        "precio_promedio": round(float(np.nanmean(P_m[:, j])), 2),
        "fechas": [d.strftime("%Y-%m-%d") for d in sim_fechas],
        "dias_revision": [int(i) for i in idx_rev],
        "demanda": r(D_m[:, j]),
        "pronostico": {"LGB": r(lgb_m[:, j]), "TSB": r(tsb_m[:, j])},
        "simulacion": simulacion,
    }
    with open(OUT / "productos" / f"{item}.json", "w") as f:
        json.dump(datos, f, separators=(",", ":"))
    indice.append({"id": item, "departamento": datos["departamento"], "abc": row["abc"],
                   "clase": row["sb_class"], "ingreso_historico": round(float(row["revenue"]), 0)})

with open(OUT / "productos.json", "w") as f:
    json.dump(indice, f, separators=(",", ":"))

meta = {"productos_simulados": len(universo), "productos_totales": len(ids_validos),
        "inicio_simulacion": sim_start.strftime("%Y-%m-%d"), "fin_simulacion": dev_end.strftime("%Y-%m-%d"),
        "semanas_simuladas": N_SIM, "revision_dias": R, "lead_time_dias": L,
        "costo_mantener_anual": H_ANUAL, "margen": MARGEN,
        "dias_imputados_pct": round(float((ventana["tipo"] == "agotado").mean() * 100), 2)}
with open(OUT / "meta_simulacion.json", "w") as f:
    json.dump(meta, f, indent=2)

tam = sum(p.stat().st_size for p in OUT.rglob("*.json")) / 1024
print(f"{len(muestra)} productos exportados | tamaño total: {tam:,.0f} KB → {OUT}")

35 productos exportados | tamaño total: 902 KB → /Users/hectormm/Documents/portfolio/public/data/m5
